In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import norm
from scipy.integrate import trapezoid
from scipy.optimize import minimize, differential_evolution

class StimulusCategoryModel:
    """
    Stimulus Category (SC) model.
    
    Agent maintains two belief distributions (one per category).
    Updates based on CHOSEN CATEGORY and feedback correctness.
    
    Parameters:
        sigma_percep: perceptual noise std
        A_repulsion: serial dependence strength
        gamma: learning rate (weight on prior vs update)
        sigma_update: width of Gaussian update kernel
    """
    
    def __init__(self, sigma_percep, A_repulsion, gamma, sigma_update,
                 x_min=-1, x_max=1, n_points=500):
        # Store parameters
        self.sigma_percep = sigma_percep
        self.A_repulsion = A_repulsion
        self.gamma = gamma
        self.sigma_update = sigma_update
        
        # Stimulus space discretisation
        self.x = np.linspace(x_min, x_max, n_points)
        
        # Initialise state
        self.reset_beliefs()
        self.s_hat_prev = None
    
    def reset_beliefs(self, p_A=None, p_B=None):
        """Reset to initial Gaussian priors or provided distributions"""
        if p_A is None:
            # Initial: A centred at -0.75, B centred at +0.75
            self.p_A = norm.pdf(self.x, loc=-0.75, scale=0.5)
            self.p_B = norm.pdf(self.x, loc=0.75, scale=0.5)
        else:
            self.p_A = p_A.copy()
            self.p_B = p_B.copy()
        self.s_hat_prev = None
    
    def _perceive_stimulus(self, s_t, rng):
        """Apply perceptual noise and repulsion"""
        # Perceptual noise
        noise = rng.normal(0, self.sigma_percep)
        s_tilde = s_t + noise
        
        # Repulsion from previous trial
        if self.s_hat_prev is not None:
            diff = s_tilde - self.s_hat_prev
            repulsion = self.A_repulsion * diff * np.exp(-np.abs(diff))
            s_hat = s_tilde + repulsion
        else:
            s_hat = s_tilde
        
        return s_hat
    
    def _find_closest_idx(self, s_hat):
        """Find index of closest element in x to s_hat"""
        return np.abs(self.x - s_hat).argmin()
    
    def _get_choice_probability(self, s_hat):
        """P(B) = p_B(s_hat) / (p_A(s_hat) + p_B(s_hat))"""
        j = self._find_closest_idx(s_hat)
        
        denom = self.p_A[j] + self.p_B[j]
        if denom <= 0:
            return 0.5
        
        p_B = self.p_B[j] / denom
        return np.clip(p_B, 1e-10, 1 - 1e-10)
    
    def _update_beliefs(self, s_hat, choice, true_category):
        """
        Update the CHOSEN category's distribution based on feedback.
        
        Args:
            s_hat: perceived stimulus
            choice: what was chosen (0=A, 1=B)
            true_category: correct answer (0=A, 1=B)
        """
        j = self._find_closest_idx(s_hat)
        
        # Gaussian kernel centred at grid point
        g = norm.pdf(self.x, loc=self.x[j], scale=self.sigma_update)
        
        correct = (choice == true_category)
        
        if choice == 0:  # Chose A → update p_A
            if correct:
                # Positive feedback: strengthen
                self.p_A = self.p_A * self.gamma + g * (1 - self.gamma)
            else:
                # Negative feedback: weaken
                self.p_A = self.p_A * self.gamma - g * (1 - self.gamma)
                min_val = np.min(self.p_A)
                if min_val < 0:
                    self.p_A = self.p_A + np.abs(min_val)
            
            # Normalise
            self.p_A = self.p_A / trapezoid(self.p_A, self.x)
        
        else:  # Chose B → update p_B
            if correct:
                self.p_B = self.p_B * self.gamma + g * (1 - self.gamma)
            else:
                self.p_B = self.p_B * self.gamma - g * (1 - self.gamma)
                min_val = np.min(self.p_B)
                if min_val < 0:
                    self.p_B = self.p_B + np.abs(min_val)
            
            self.p_B = self.p_B / trapezoid(self.p_B, self.x)
        
        # Store for next trial's repulsion
        self.s_hat_prev = s_hat
    
    def simulate_session(self, stimuli, categories, no_response=None, rng=None):
        """
        Simulate choices for a session.
        
        Model generates choices and updates based on MODEL's choices.
        """
        if rng is None:
            rng = np.random.default_rng()
        if no_response is None:
            no_response = np.zeros(len(stimuli), dtype=bool)
        
        n = len(stimuli)
        choices = np.full(n, np.nan)
        p_B_sequence = np.full(n, np.nan)
        
        for t in range(n):
            if no_response[t]:
                continue
            
            # Perceive
            s_hat = self._perceive_stimulus(stimuli[t], rng)
            
            # Compute choice probability
            p_B = self._get_choice_probability(s_hat)
            p_B_sequence[t] = p_B
            
            # Model makes choice
            choice = rng.binomial(1, p_B)
            choices[t] = choice
            
            # Update based on MODEL's choice
            self._update_beliefs(s_hat, choice, categories[t])
        
        return choices, p_B_sequence
    
    def compute_log_likelihood(self, stimuli, categories, observed_choices,
                                no_response=None, seed=42):
        """
        Compute log-likelihood of observed (animal) choices.
        
        Updates based on ANIMAL's choices (not model's).
        """
        rng = np.random.default_rng(seed)
        if no_response is None:
            no_response = np.isnan(observed_choices)
        
        log_lik = []
        
        for t in range(len(stimuli)):
            if no_response[t]:
                continue
            
            # Perceive
            s_hat = self._perceive_stimulus(stimuli[t], rng)
            
            # Compute choice probability
            p_B = self._get_choice_probability(s_hat)
            
            # Log-likelihood of animal's choice
            animal_choice = int(observed_choices[t])
            if animal_choice == 1:
                log_lik.append(np.log(p_B))
            else:
                log_lik.append(np.log(1 - p_B))
            
            # Update based on ANIMAL's choice (tracking animal's belief state)
            self._update_beliefs(s_hat, animal_choice, categories[t])
        
        return np.array(log_lik)
    
    def get_beliefs_copy(self):
        """Return copy of current beliefs"""
        return self.p_A.copy(), self.p_B.copy()
    
    def set_beliefs(self, p_A, p_B):
        """Set beliefs (for carrying across sessions)"""
        self.p_A = p_A.copy()
        self.p_B = p_B.copy()
        
    def get_params(self):
        return {
            'sigma_percep': self.sigma_percep,
            'A_repulsion': self.A_repulsion,
            'gamma': self.gamma,
            'sigma_update': self.sigma_update
        }
    
    # ==================== CLASS METHODS FOR FITTING ====================
    
    @classmethod
    def get_bounds(cls):
        """Parameter bounds for fitting"""
        return {
            'sigma_percep': (0.05, 0.3),  
            'A_repulsion': (0.0, 0.5),    
            'mu_learning': (0.1, 0.9),    
            'mu_relax': (0.05, 0.4)    
		}
    
    @classmethod
    def get_param_names(cls):
        return ['sigma_percep', 'A_repulsion', 'gamma', 'sigma_update']
    
    @classmethod
    def fit(cls, stimuli, categories, observed_choices, no_response=None,
            fixed_params=None, initial_beliefs=None,
            method='L-BFGS-B', n_restarts=5, seed=42):
        """
        Fit model to observed data using MLE.
        
        Args:
            fixed_params: dict of params to fix
            initial_beliefs: tuple (p_A, p_B) for multi-session
        """
        all_bounds = cls.get_bounds()
        all_param_names = cls.get_param_names()
        
        if fixed_params is None:
            fixed_params = {}
        
        free_param_names = [p for p in all_param_names if p not in fixed_params]
        free_bounds = [all_bounds[p] for p in free_param_names]
        
        rng = np.random.default_rng(seed)
        
        def neg_log_likelihood(free_param_values):
            params = fixed_params.copy()
            for name, val in zip(free_param_names, free_param_values):
                params[name] = val
            
            model = cls(
                sigma_percep=params['sigma_percep'],
                A_repulsion=params['A_repulsion'],
                gamma=params['gamma'],
                sigma_update=params['sigma_update']
            )
            
            if initial_beliefs is not None:
                model.set_beliefs(*initial_beliefs)
            
            ll = model.compute_log_likelihood(
                stimuli, categories, observed_choices,
                no_response=no_response
            )
            sum_ll = np.sum(ll)
            
            return -sum_ll
        
        best_nll = np.inf
        best_free_params = None
        
        for i in range(n_restarts):
            x0 = [rng.uniform(b[0], b[1]) for b in free_bounds]
            
            try:
                result = minimize(
                    neg_log_likelihood,
                    x0=x0,
                    method=method,
                    bounds=free_bounds,
                    options={'maxiter': 1000}
                )
                
                if result.fun < best_nll:
                    best_nll = result.fun
                    best_free_params = result.x
            
            except Exception:
                continue
        
        if best_free_params is None:
            raise RuntimeError("All optimisation attempts failed")
        
        best_params = fixed_params.copy()
        for name, val in zip(free_param_names, best_free_params):
            best_params[name] = val
        
        best_model = cls(**best_params)
        
        if initial_beliefs is not None:
            best_model.set_beliefs(*initial_beliefs)
        
        n_valid = np.sum(~np.isnan(observed_choices)) if no_response is None else np.sum(~no_response)
        n_free_params = len(free_param_names)
        aic = 2 * best_nll + 2 * n_free_params
        bic = 2 * best_nll + n_free_params * np.log(n_valid)
        
        results = {
            'params': best_params,
            'fixed_params': fixed_params,
            'free_params': dict(zip(free_param_names, best_free_params)),
            'nll': best_nll,
            'aic': aic,
            'bic': bic,
            'n_trials': n_valid,
            'n_free_params': n_free_params
        }
        
        return best_model, results

    @classmethod
    def fit_sessions(cls, sessions, fixed_params=None, carry_beliefs=False,
                     verbose=True, **fit_kwargs):
        """Fit model to multiple sessions."""
        models = []
        results = []
        current_beliefs = None
        
        for i, session in enumerate(sessions):
            if verbose:
                print(f"Fitting session {i+1}/{len(sessions)}...", end=' ')
            
            try:
                model, res = cls.fit(
                    session['stimuli'],
                    session['categories'],
                    session['choices'],
                    session.get('no_response', None),
                    fixed_params=fixed_params,
                    initial_beliefs=current_beliefs if carry_beliefs else None,
                    **fit_kwargs
                )
                
                res['session_idx'] = i
                res['success'] = True
                
                models.append(model)
                results.append(res)
                
                if carry_beliefs:
                    model.reset_beliefs(*(current_beliefs if current_beliefs else (None, None)))
                    model.compute_log_likelihood(
                        session['stimuli'],
                        session['categories'],
                        session['choices'],
                        session.get('no_response', None)
                    )
                    current_beliefs = model.get_beliefs_copy()
                
                if verbose:
                    print(f"NLL={res['nll']:.1f}")
            
            except Exception as e:
                if verbose:
                    print(f"FAILED: {e}")
                models.append(None)
                results.append({'session_idx': i, 'success': False})
        
        trajectories = cls._extract_trajectories(results)
        
        return models, results, trajectories
    
    @classmethod
    def _extract_trajectories(cls, results):
        param_names = cls.get_param_names()
        
        trajectories = {name: [] for name in param_names}
        trajectories['session'] = []
        trajectories['nll'] = []
        trajectories['aic'] = []
        
        for r in results:
            if r.get('success', False):
                trajectories['session'].append(r['session_idx'])
                trajectories['nll'].append(r['nll'])
                trajectories['aic'].append(r['aic'])
                for name in param_names:
                    trajectories[name].append(r['params'][name])
        
        for key in trajectories:
            trajectories[key] = np.array(trajectories[key])
        
        return trajectories
    
    @classmethod
    def plot_trajectories(cls, trajectories, figsize=(12, 8)):
        param_names = cls.get_param_names()
        
        fig, axes = plt.subplots(2, 2, figsize=figsize)
        axes = axes.flatten()
        
        sessions = trajectories['session']
        
        for i, name in enumerate(param_names):
            ax = axes[i]
            ax.plot(sessions, trajectories[name], 'o-', markersize=4)
            ax.set_xlabel('Session')
            ax.set_ylabel(name)
            ax.set_title(name)
            ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        return fig
    
    @classmethod
    def parameter_recovery(cls, n_tests=20, n_trials=300, fixed_params=None, seed=0):
        all_bounds = cls.get_bounds()
        all_param_names = cls.get_param_names()
        
        if fixed_params is None:
            fixed_params = {}
        
        free_param_names = [p for p in all_param_names if p not in fixed_params]
        
        rng = np.random.default_rng(seed)
        
        true_params = {name: [] for name in free_param_names}
        recovered_params = {name: [] for name in free_param_names}
        
        for i in range(n_tests):
            print(f"Recovery test {i+1}/{n_tests}...", end=' ')
            
            true_free = {}
            for name in free_param_names:
                true_free[name] = rng.uniform(*all_bounds[name])
            
            all_true = {**fixed_params, **true_free}
            
            stimuli = rng.uniform(-1, 1, n_trials)
            categories = (stimuli > 0).astype(int)
            
            model = cls(**all_true)
            sim_choices, _ = model.simulate_session(stimuli, categories, rng=rng)
            
            try:
                _, results = cls.fit(
                    stimuli, categories, sim_choices,
                    fixed_params=fixed_params,
                    n_restarts=3
                )
                
                for name in free_param_names:
                    true_params[name].append(true_free[name])
                    recovered_params[name].append(results['params'][name])
                
                print("OK")
            
            except Exception as e:
                print(f"FAILED: {e}")
        
        for name in free_param_names:
            true_params[name] = np.array(true_params[name])
            recovered_params[name] = np.array(recovered_params[name])
        
        correlations = {}
        for name in free_param_names:
            correlations[name] = np.corrcoef(
                true_params[name],
                recovered_params[name]
            )[0, 1]
        
        return true_params, recovered_params, correlations
    
    @classmethod
    def plot_recovery(cls, true_params, recovered_params, correlations):
        param_names = list(true_params.keys())
        n_params = len(param_names)
        
        cols = min(n_params, 2)
        rows = (n_params + cols - 1) // cols
        
        fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
        if n_params == 1:
            axes = [axes]
        else:
            axes = axes.flatten()
        
        bounds = cls.get_bounds()
        
        for i, name in enumerate(param_names):
            ax = axes[i]
            
            ax.scatter(true_params[name], recovered_params[name], alpha=0.6)
            
            lims = list(bounds[name])
            ax.plot(lims, lims, 'k--', alpha=0.5)
            
            ax.set_title(f'{name}\nr = {correlations[name]:.3f}')
            ax.set_xlabel('True')
            ax.set_ylabel('Recovered')
            ax.set_xlim(lims)
            ax.set_ylim(lims)
            ax.set_aspect('equal')
        
        for i in range(n_params, len(axes)):
            axes[i].set_visible(False)
        
        plt.tight_layout()
        return fig

In [ ]:
#load data

In [ ]:
# Load your data (you'll need to adapt this to your data format)
stimuli = session_data['stimuli']           # array of stimulus values
categories = session_data['categories']      # array of true categories (0/1)
observed_choices = session_data['choices']   # array of animal choices (0/1)
no_response = session_data['no_response']    # boolean array

In [ ]:
# 1. Fit single session
model, results = StimulusCategoryModel.fit(stimuli, categories, observed_choices)
print(results['params'])

# 2. Fit with fixed parameters (for multi-session)
model, results = StimulusCategoryModel.fit(
    stimuli, categories, observed_choices,
    fixed_params={'sigma_percep': 0.15, 'A_repulsion': 0.1}
)
print(f"Free params: {results['free_params']}")

# 3. Fit all sessions
models, results, trajectories = StimulusCategoryModel.fit_sessions(
    sessions,
    fixed_params={'sigma_percep': 0.15, 'A_repulsion': 0.1},
    carry_belief=True
)
StimulusCategoryModel.plot_trajectories(trajectories)
plt.show()

# 4. Parameter recovery
true_p, rec_p, corrs = StimulusCategoryModel.parameter_recovery(
    n_tests=30,
    fixed_params={'sigma_percep': 0.15, 'A_repulsion': 0.1}
)
BoundaryEstimationModel.plot_recovery(true_p, rec_p, corrs)
plt.show()